## Introduction

TBD

 ## Python Imports

In [1]:
# Standard Library
import os
import sys
import warnings

from pathlib import Path

# Data Handling
import pandas as pd

# Suppress warnings
warnings.filterwarnings("ignore")

In [2]:
# Add the source subdirectory to the system path to allow import config from settings.py
current_directory = Path(os.getcwd())
website_base_directory = current_directory.parent.parent.parent
src_directory = website_base_directory / "src"
sys.path.append(str(src_directory)) if str(src_directory) not in sys.path else None

# Import settings.py
from settings import config

# Add configured directories from config to path
SOURCE_DIR = config("SOURCE_DIR")
sys.path.append(str(Path(SOURCE_DIR))) if str(Path(SOURCE_DIR)) not in sys.path else None

# Add other configured directories
BASE_DIR = config("BASE_DIR")
CONTENT_DIR = config("CONTENT_DIR")
POSTS_DIR = config("POSTS_DIR")
PAGES_DIR = config("PAGES_DIR")
PUBLIC_DIR = config("PUBLIC_DIR")
SOURCE_DIR = config("SOURCE_DIR")
DATA_DIR = config("DATA_DIR")
DATA_MANUAL_DIR = config("DATA_MANUAL_DIR")

## Python Functions

Here are the functions needed for this project:

* [load_data](/posts/reusable-extensible-python-functions-financial-data-analysis/#load_data): Load data from a CSV, Excel, or Pickle file into a pandas DataFrame.

In [3]:
from load_data import load_data

## Set Variables

In [7]:
ticker = "UXI"
source = "Polygon"
asset_class = "Exchange_Traded_Funds"
timeframe = "day"

## Load Post-Split Data

In [8]:
df_post_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-postsplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

df_post_split

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.000000,34.5128,38,None
1,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.000000,34.1661,25,None
2,2024-04-08 04:00:00,34.4401,34.4401,34.2902,34.2902,1462.000000,34.4235,28,None
3,2024-04-09 04:00:00,33.6000,34.0908,33.5900,34.0908,778.000000,33.7406,21,None
4,2024-04-10 04:00:00,33.6300,33.6900,33.3753,33.6033,3256.000000,33.5345,39,None
...,...,...,...,...,...,...,...,...,...
543,2026-06-04 04:00:00,57.1600,57.6438,57.1600,57.6438,166.090988,57.2811,41,None
544,2026-06-05 04:00:00,57.3750,57.3750,56.0400,56.4510,2948.892254,56.6844,68,None
545,2026-06-08 04:00:00,56.2500,56.4300,56.0076,56.0076,888.788869,56.4070,46,None
546,2026-06-09 04:00:00,57.7250,57.7250,56.0500,57.2026,2629.444024,56.7753,76,None


## Calc Cutoff and Split Check Dates

In [9]:
cutoff_date = df_post_split.loc[3, "Date"]
print(f"Cutoff date for split adjustment: {cutoff_date}")
split_check_date = df_post_split.loc[4, "Date"]
print(f"Split check date: {split_check_date}")

Cutoff date for split adjustment: 2024-04-09 04:00:00
Split check date: 2024-04-10 04:00:00


## Load Pre-Split Data

In [10]:
df_pre_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-presplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

print(df_pre_split.shape)
df_pre_split[df_pre_split["Date"] < cutoff_date]

(610, 9)


,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.5589,27.5589,27.5589,27.5589,283.0,27.5347,28,None
1,2023-08-01 04:00:00,27.7500,27.7891,27.7390,27.7891,698.0,27.7348,14,None
2,2023-08-02 04:00:00,27.2800,27.5300,27.2034,27.2034,52597.0,27.5290,13,None
3,2023-08-03 04:00:00,26.8300,26.8810,26.8300,26.8810,334.0,26.7248,14,None
4,2023-08-04 04:00:00,26.7590,26.7590,26.4599,26.4599,316.0,26.8659,27,None
...,...,...,...,...,...,...,...,...,...
169,2024-04-02 04:00:00,33.7900,34.0158,33.7670,33.7670,5651.0,33.8582,38,None
170,2024-04-03 04:00:00,34.2300,34.3000,34.1138,34.1138,2017.0,34.2324,24,None
171,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None
172,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.0,34.1661,25,None


## Calc Split Ratio

In [11]:
# Extract "open" price from the same row from both pre and post split dataframes to calculate the split ratio
pre_split_open_price = df_pre_split[df_pre_split["Date"] == split_check_date]["open"].values[0]
post_split_open_price = df_post_split[df_post_split["Date"] == split_check_date]["open"].values[0]
SPLIT_RATIO = pre_split_open_price / post_split_open_price
print(f"Calculated split ratio: {SPLIT_RATIO}")

Calculated split ratio: 1.0


## Calc Corrected Pre-Split Data

In [12]:
df_pre_split_corrected = df_pre_split.copy()
df_pre_split_corrected["open"] = df_pre_split_corrected["open"] / SPLIT_RATIO
df_pre_split_corrected["high"] = df_pre_split_corrected["high"] / SPLIT_RATIO
df_pre_split_corrected["low"] = df_pre_split_corrected["low"] / SPLIT_RATIO
df_pre_split_corrected["close"] = df_pre_split_corrected["close"] / SPLIT_RATIO
df_pre_split_corrected["volume"] = df_pre_split_corrected["volume"] * SPLIT_RATIO
df_pre_split_corrected["vwap"] = df_pre_split_corrected["vwap"] / SPLIT_RATIO
df_pre_split_corrected[df_pre_split_corrected["Date"] < cutoff_date]

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.5589,27.5589,27.5589,27.5589,283.0,27.5347,28,None
1,2023-08-01 04:00:00,27.7500,27.7891,27.7390,27.7891,698.0,27.7348,14,None
2,2023-08-02 04:00:00,27.2800,27.5300,27.2034,27.2034,52597.0,27.5290,13,None
3,2023-08-03 04:00:00,26.8300,26.8810,26.8300,26.8810,334.0,26.7248,14,None
4,2023-08-04 04:00:00,26.7590,26.7590,26.4599,26.4599,316.0,26.8659,27,None
...,...,...,...,...,...,...,...,...,...
169,2024-04-02 04:00:00,33.7900,34.0158,33.7670,33.7670,5651.0,33.8582,38,None
170,2024-04-03 04:00:00,34.2300,34.3000,34.1138,34.1138,2017.0,34.2324,24,None
171,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None
172,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.0,34.1661,25,None


## Combine Pre- and Post-Split Data

In [13]:
display_date = df_post_split.loc[5, "Date"]
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True)
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
166,2024-03-27 04:00:00,34.0400,34.6300,34.0300,34.6290,1153.0,34.1215,23,None
167,2024-03-28 04:00:00,34.8400,34.8400,34.3799,34.6901,4193.0,34.6179,44,None
168,2024-04-01 04:00:00,35.2900,35.2900,33.7701,34.1643,6737.0,34.3749,42,None
169,2024-04-02 04:00:00,33.7900,34.0158,33.7670,33.7670,5651.0,33.8582,38,None
170,2024-04-03 04:00:00,34.2300,34.3000,34.1138,34.1138,2017.0,34.2324,24,None
171,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None
172,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.0,34.1661,25,None
173,2024-04-08 04:00:00,34.4401,34.4401,34.2902,34.2902,1462.0,34.4235,28,None
174,2024-04-09 04:00:00,33.6000,34.0908,33.5900,34.0908,778.0,33.7406,21,None
175,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None


## Confirm Rows To Be Dropped

In [14]:
print(f"Should drop the following duplicate rows if they exist:")
combined[combined.round(3).duplicated(subset=["Date", "open", "high", "low", "close", "volume"], keep=False)]

Should drop the following duplicate rows if they exist:


,Date,open,high,low,close,volume,vwap,transactions,otc
171,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None
172,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.0,34.1661,25,None
173,2024-04-08 04:00:00,34.4401,34.4401,34.2902,34.2902,1462.0,34.4235,28,None
174,2024-04-09 04:00:00,33.6000,34.0908,33.5900,34.0908,778.0,33.7406,21,None
175,2024-04-04 04:00:00,34.6500,34.6500,33.5199,33.5199,1765.0,34.5128,38,None
176,2024-04-05 04:00:00,34.3900,34.4752,34.3900,34.4752,613.0,34.1661,25,None
177,2024-04-08 04:00:00,34.4401,34.4401,34.2902,34.2902,1462.0,34.4235,28,None
178,2024-04-09 04:00:00,33.6000,34.0908,33.5900,34.0908,778.0,33.7406,21,None


## Drop Duplicate Rows

In [15]:
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True).round(3).drop_duplicates(subset=["Date", "open", "high", "low", "close", "volume"], keep="last")
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
162,2024-03-21 04:00:00,34.45,34.647,34.195,34.574,8667.0,34.384,93,None
163,2024-03-22 04:00:00,34.99,34.990,34.341,34.380,5436.0,34.495,74,None
164,2024-03-25 04:00:00,34.25,34.250,33.839,33.839,5811.0,34.066,88,None
165,2024-03-26 04:00:00,34.04,34.040,33.632,33.632,4211.0,33.757,41,None
166,2024-03-27 04:00:00,34.04,34.630,34.030,34.629,1153.0,34.122,23,None
167,2024-03-28 04:00:00,34.84,34.840,34.380,34.690,4193.0,34.618,44,None
168,2024-04-01 04:00:00,35.29,35.290,33.770,34.164,6737.0,34.375,42,None
169,2024-04-02 04:00:00,33.79,34.016,33.767,33.767,5651.0,33.858,38,None
170,2024-04-03 04:00:00,34.23,34.300,34.114,34.114,2017.0,34.232,24,None
175,2024-04-04 04:00:00,34.65,34.650,33.520,33.520,1765.0,34.513,38,None


## Reset Index And Display Data

In [16]:
combined.reset_index(drop=True, inplace=True)
combined

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.559,27.559,27.559,27.559,283.000,27.535,28,None
1,2023-08-01 04:00:00,27.750,27.789,27.739,27.789,698.000,27.735,14,None
2,2023-08-02 04:00:00,27.280,27.530,27.203,27.203,52597.000,27.529,13,None
3,2023-08-03 04:00:00,26.830,26.881,26.830,26.881,334.000,26.725,14,None
4,2023-08-04 04:00:00,26.759,26.759,26.460,26.460,316.000,26.866,27,None
...,...,...,...,...,...,...,...,...,...
714,2026-06-04 04:00:00,57.160,57.644,57.160,57.644,166.091,57.281,41,None
715,2026-06-05 04:00:00,57.375,57.375,56.040,56.451,2948.892,56.684,68,None
716,2026-06-08 04:00:00,56.250,56.430,56.008,56.008,888.789,56.407,46,None
717,2026-06-09 04:00:00,57.725,57.725,56.050,57.203,2629.444,56.775,76,None


## Export

In [17]:
# Export the combined DataFrame to a pickle file
combined.to_pickle(DATA_DIR / source / asset_class / timeframe / f"{ticker}.pkl")